> **Note:** This notebook only reads from the CSV file for matching purposes. It never modifies or writes to the CSV file.

# Check context_quote matches in PDF
This notebook checks if each `context_quote` from the processed CSV appears in the original PDF, and highlights matches in a new PDF.

In [14]:
import pandas as pd
import fitz  # PyMuPDF
import os
import re
from rapidfuzz import fuzz  # for improved fuzzy matching

# Paths
csv_path = '/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Processed_Results_V1/claude_v1_49_processed.csv'
pdf_path = '/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Documents/49.pdf'
output_pdf_path = '/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Check_doc/49_highlighted.pdf'

# Load CSV (read-only, never modify or write to the CSV)
df = pd.read_csv(csv_path)

# Open PDF
doc = fitz.open(pdf_path)

def normalize_text(text):
    # Remove all non-alphanumeric except spaces, lowercase, and collapse whitespace
    return re.sub(r'[^a-z0-9 ]+', '', str(text).lower().replace('\n', ' ')).strip()

def sliding_window_fuzzy_match(quote, page_text, threshold=80, window_size=400, step=50):
    """
    For long context quotes, slide a window over the page text and compute fuzzy match.
    Returns True if any window matches above the threshold.
    """
    quote_norm = normalize_text(quote)
    page_words = normalize_text(page_text).split()
    if len(page_words) <= window_size:
        score = fuzz.partial_ratio(quote_norm, ' '.join(page_words))
        return score >= threshold
    for i in range(0, len(page_words) - window_size + 1, step):
        window = ' '.join(page_words[i:i+window_size])
        score = fuzz.partial_ratio(quote_norm, window)
        if score >= threshold:
            return True
    return False

results = []
for idx, row in enumerate(df.itertuples(index=False)):
    quote = str(row.context_quote).strip()
    found = False
    for page_num in range(len(doc)):
        page = doc[page_num]
        page_text = page.get_text()
        # Use sliding window fuzzy match for robust matching
        if sliding_window_fuzzy_match(quote, page_text):
            found = True
            # Try to highlight the exact quote if possible
            text_instances = page.search_for(quote)
            if not text_instances and len(quote) > 80:
                text_instances = page.search_for(quote[:80])
            for inst in text_instances:
                page.add_highlight_annot(inst)
            break
    results.append({'row': idx+2, 'context_quote': quote, 'match': found})

doc.save(output_pdf_path, garbage=4, deflate=True)

for r in results:
    print(f"Row {r['row']}: {'MATCH' if r['match'] else 'NO MATCH'}")
    if not r['match']:
        print(f"  Quote: {r['context_quote'][:120]}...")

print(f'Highlighted PDF saved to: {output_pdf_path}')

Row 2: NO MATCH
  Quote: The proposed Project would allow Sunrise Wind to construct, operate, maintain, and eventually decommission a wind energy...
Row 3: MATCH
Row 4: MATCH
Row 5: MATCH
Row 6: MATCH
Row 7: MATCH
Row 8: MATCH
Row 9: MATCH
Row 10: MATCH
Row 11: MATCH
Row 12: MATCH
Row 13: MATCH
Row 14: MATCH
Row 15: NO MATCH
  Quote: It is anticipated that 15 vessels would be required for OCS-DC installation....
Row 16: MATCH
Row 17: MATCH
Highlighted PDF saved to: /Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Check_doc/49_highlighted.pdf
